# Simple perceptron for joint $N_{\mathrm{part}}$ and $b$ regression

This notebook trains a small one-hidden-layer perceptron on UrQMD event-level features built in place after removing protons and antiprotons. It predicts two continuous targets jointly:

- participant count, $N_{\mathrm{part}}$
- impact parameter, $b$ [fm]

The workflow is capped at **10,000 unique events**: 8,000 training, 1,000 validation, and 1,000 test events. The notebook is saved without execution outputs.

## 1. Imports and configuration

In [ ]:
from copy import deepcopy
from pathlib import Path

import awkward as ak
import matplotlib.pyplot as plt
import numpy as np
import torch
import uproot
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
SEED = 42
MAX_EVENTS = 10_000
N_TRAIN = 8_000
N_VAL = 1_000
N_TEST = 1_000
BATCH_SIZE = 256
EPOCHS = 80
LEARNING_RATE = 1e-3
HIDDEN_DIM = 32
PATIENCE = 10
EXCLUDED_ABS_PDG = 2212

assert N_TRAIN + N_VAL + N_TEST <= MAX_EVENTS
np.random.seed(SEED)
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Device:", device)

## 2. Build a proton-free UrQMD table

The event summaries are rebuilt directly from the raw particle arrays. Protons and antiprotons are removed before multiplicity or momentum statistics are calculated. `b` is used only as the second target; it is not included in the feature matrix.

In [ ]:
relative_data_path = Path("Data/combined_100k.root")
data_path = next(
    (
        root / relative_data_path
        for root in (Path.cwd(), *Path.cwd().parents)
        if (root / relative_data_path).is_file()
    ),
    None,
)

if data_path is None:
    raise FileNotFoundError(f"Could not locate {relative_data_path} from the project tree.")

with uproot.open(data_path) as root_file:
    tree = root_file["urqmd"]
    if tree.num_entries < MAX_EVENTS:
        raise ValueError(f"ROOT tree contains only {tree.num_entries:,} events.")
    events = tree.arrays(
        ["pid", "px", "py", "pz", "Npart", "b"],
        entry_start=0,
        entry_stop=MAX_EVENTS,
        library="ak",
    )

particle_mask = abs(events["pid"]) != EXCLUDED_ABS_PDG
px = events["px"][particle_mask]
py = events["py"][particle_mask]
pz = events["pz"][particle_mask]
pt = np.sqrt(px**2 + py**2)
eta = np.arcsinh(pz / ak.where(pt > 0, pt, 1.0))

def filled_stat(values, operation):
    result = operation(values, axis=1, mask_identity=True)
    return np.asarray(ak.to_numpy(ak.fill_none(result, 0.0)), dtype=np.float32)


feature_names = np.array([
    "n_particles", "sum_px", "sum_py", "sum_pz", "sum_pt",
    "mean_pt", "std_pt", "mean_abs_eta", "std_abs_eta",
])
X_all = np.column_stack([
    ak.to_numpy(ak.sum(particle_mask, axis=1)),
    filled_stat(px, ak.sum),
    filled_stat(py, ak.sum),
    filled_stat(pz, ak.sum),
    filled_stat(pt, ak.sum),
    filled_stat(pt, ak.mean),
    filled_stat(pt, ak.std),
    filled_stat(abs(eta), ak.mean),
    filled_stat(abs(eta), ak.std),
]).astype(np.float32)
npart_all = ak.to_numpy(events["Npart"]).astype(np.float32)
b_all = ak.to_numpy(events["b"]).astype(np.float32)

permutation = rng.permutation(MAX_EVENTS)
train_idx_full = permutation[:N_TRAIN]
val_idx_full = permutation[N_TRAIN : N_TRAIN + N_VAL]
test_idx_full = permutation[N_TRAIN + N_VAL :]

targets_all = np.column_stack([npart_all, b_all]).astype(np.float32)
target_names = np.array(["Npart", "b [fm]"])

assert len(X_all) == MAX_EVENTS
assert np.all(np.isfinite(X_all))
print(f"Built {len(X_all):,} proton-free events from {data_path}")
print("Features:", ", ".join(feature_names))
print("Targets:", ", ".join(target_names))

## 3. Construct capped, isolated splits

Rows are sampled independently from deterministic train/validation/test partitions. Feature and target standardization statistics are calculated from the capped training subset only.

In [ ]:
def sample_without_replacement(indices, size):
    if len(indices) < size:
        raise ValueError(f"Cannot sample {size:,} rows from a split of {len(indices):,} rows.")
    return rng.choice(indices, size=size, replace=False)


train_idx = sample_without_replacement(train_idx_full, N_TRAIN)
val_idx = sample_without_replacement(val_idx_full, N_VAL)
test_idx = sample_without_replacement(test_idx_full, N_TEST)

selected_idx = np.concatenate([train_idx, val_idx, test_idx])
assert len(selected_idx) <= MAX_EVENTS
assert len(np.unique(selected_idx)) == len(selected_idx)

feature_mean = X_all[train_idx].mean(axis=0)
feature_std = X_all[train_idx].std(axis=0)
feature_std = np.where(feature_std > 1e-8, feature_std, 1.0)

target_mean = targets_all[train_idx].mean(axis=0)
target_std = targets_all[train_idx].std(axis=0)
target_std = np.where(target_std > 1e-8, target_std, 1.0)

def standardized_rows(indices):
    x = (X_all[indices] - feature_mean) / feature_std
    y = (targets_all[indices] - target_mean) / target_std
    return x.astype(np.float32), y.astype(np.float32)


X_train, y_train = standardized_rows(train_idx)
X_val, y_val = standardized_rows(val_idx)
X_test, y_test = standardized_rows(test_idx)

print(f"Train: {len(train_idx):,}; validation: {len(val_idx):,}; test: {len(test_idx):,}")
print(f"Total unique events: {len(selected_idx):,} / {MAX_EVENTS:,}")
print("Training target mean:", dict(zip(target_names, target_mean.round(3))))
print("Training target std:", dict(zip(target_names, target_std.round(3))))

In [ ]:
def make_loader(x, y, shuffle=False):
    dataset = TensorDataset(torch.from_numpy(x), torch.from_numpy(y))
    generator = torch.Generator().manual_seed(SEED)
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        generator=generator if shuffle else None,
        num_workers=0,
    )


train_loader = make_loader(X_train, y_train, shuffle=True)
val_loader = make_loader(X_val, y_val)
test_loader = make_loader(X_test, y_test)

## 4. Define and train the perceptron

This is deliberately small: one shared hidden layer and a two-neuron regression head. Standardizing both targets prevents the larger numerical scale of $N_{\mathrm{part}}$ from dominating the loss.

In [ ]:
class JointPerceptron(nn.Module):
    def __init__(self, n_features, hidden_dim=32):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(n_features, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, x):
        return self.network(x)


model = JointPerceptron(X_train.shape[1], HIDDEN_DIM).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_fn = nn.MSELoss()

print(model)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
def epoch_loss(loader, training):
    model.train(training)
    total_loss = 0.0
    total_events = 0

    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for features, targets in loader:
            features = features.to(device)
            targets = targets.to(device)

            if training:
                optimizer.zero_grad(set_to_none=True)

            predictions = model(features)
            loss = loss_fn(predictions, targets)

            if training:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * len(features)
            total_events += len(features)

    return total_loss / total_events


history = {"train": [], "validation": []}
best_state = None
best_val_loss = np.inf
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    train_loss = epoch_loss(train_loader, training=True)
    val_loss = epoch_loss(val_loader, training=False)
    history["train"].append(train_loss)
    history["validation"].append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch == 1 or epoch % 10 == 0:
        print(f"Epoch {epoch:3d}: train={train_loss:.5f}, validation={val_loss:.5f}")

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping at epoch {epoch}")
        break

model.load_state_dict(best_state)
print(f"Best standardized validation MSE: {best_val_loss:.5f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history["train"], label="Train")
ax.plot(history["validation"], label="Validation")
ax.set(xlabel="Epoch", ylabel="Standardized MSE", title="Perceptron training history")
ax.legend()
plt.show()

## 5. Evaluate each target in physical units

In [ ]:
def predict_physical(loader):
    model.eval()
    standardized_predictions = []
    standardized_targets = []

    with torch.no_grad():
        for features, targets in loader:
            standardized_predictions.append(model(features.to(device)).cpu().numpy())
            standardized_targets.append(targets.numpy())

    predictions = np.concatenate(standardized_predictions) * target_std + target_mean
    targets = np.concatenate(standardized_targets) * target_std + target_mean
    return targets, predictions


def metrics_by_target(y_true, y_pred):
    results = {}
    for column, name in enumerate(target_names):
        residual = y_pred[:, column] - y_true[:, column]
        mae = np.mean(np.abs(residual))
        rmse = np.sqrt(np.mean(residual**2))
        denominator = np.sum((y_true[:, column] - y_true[:, column].mean()) ** 2)
        r2 = 1.0 - np.sum(residual**2) / denominator
        results[name] = {"MAE": mae, "RMSE": rmse, "R2": r2}
    return results


val_true, val_pred = predict_physical(val_loader)
test_true, test_pred = predict_physical(test_loader)

for split_name, split_metrics in {
    "Validation": metrics_by_target(val_true, val_pred),
    "Test": metrics_by_target(test_true, test_pred),
}.items():
    print(split_name)
    for target_name, values in split_metrics.items():
        formatted = ", ".join(f"{key}={value:.4f}" for key, value in values.items())
        print(f"  {target_name}: {formatted}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for column, (ax, name) in enumerate(zip(axes, target_names)):
    ax.hexbin(val_true[:, column], val_pred[:, column], gridsize=35, mincnt=1, cmap="viridis")
    limits = [
        min(val_true[:, column].min(), val_pred[:, column].min()),
        max(val_true[:, column].max(), val_pred[:, column].max()),
    ]
    ax.plot(limits, limits, "r--", linewidth=1, label="Ideal")
    ax.set(xlabel=f"True {name}", ylabel=f"Predicted {name}", title=f"Validation: {name}")
    ax.legend()

fig.tight_layout()
plt.show()

## Interpretation notes

- Predicting `b` from final-state observables is scientifically different from feeding `b` into an $N_{\mathrm{part}}$ predictor; here it is a target, so there is no geometry leakage into the inputs.
- Every event summary is calculated only after proton and antiproton particles are removed.
- Compare the two targets separately because their physical scales and irreducible fluctuations differ.
- Increase `MAX_EVENTS` only after the capped workflow behaves as expected.